# Chapter 7 - Bài Tập
**Working with Keras: A deep dive**

Mục tiêu thực hành:

- Phân biệt các mức workflow trong Keras: API mức cao, Functional API, subclassing và custom training loop.
- Xây dựng mô hình bằng `Sequential`, Functional API và kế thừa `keras.Model`.
- Sử dụng `compile()`, `fit()`, `evaluate()`, `predict()`, custom metric và callback.
- Tự viết training/evaluation loop bằng `tf.GradientTape`, `tf.function` và `train_step()` tùy biến.

Dữ liệu trong notebook là dữ liệu giả lập nhỏ để sinh viên tập trung vào API Keras thay vì xử lý dữ liệu phức tạp.

## Hướng dẫn làm bài

1. Chạy lần lượt các cell từ trên xuống dưới.
2. Với các cell có `TODO`, hãy hoàn thiện phần còn thiếu trước khi chạy tiếp.
3. Với các câu hỏi lý thuyết, trả lời trực tiếp trong cell Markdown được đánh dấu **Câu trả lời của sinh viên**.
4. Không cần tải dữ liệu từ Internet.
5. Khi nộp bài, notebook cần có output của các cell đã chạy và phần trả lời lý thuyết.

## Chuẩn bị môi trường

Nếu máy chưa có TensorFlow, sinh viên cần cài đặt TensorFlow trước khi chạy notebook. Các bài tập bên dưới giả định môi trường có TensorFlow/Keras 2.x hoặc 3.x.

In [6]:
import os
import shutil

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Cố định seed để kết quả ngẫu nhiên dễ tái lập khi chạy lại notebook.
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## Tạo dữ liệu giả lập

Notebook dùng ba nhóm dữ liệu:

- Dữ liệu phân loại nhiều lớp dạng vector.
- Dữ liệu regression nhỏ để viết custom metric.
- Dữ liệu ticket hỗ trợ khách hàng, mô phỏng bài toán nhiều input và nhiều output.

In [7]:
# Dữ liệu phân loại vector: mỗi mẫu có 20 feature và thuộc 1 trong 3 lớp.
num_samples = 1500
num_features = 20
num_classes = 3

# Sinh dữ liệu đầu vào theo phân phối chuẩn.
x = np.random.normal(size=(num_samples, num_features)).astype('float32')

# true_w là bộ trọng số ẩn dùng để tạo nhãn có quy luật.
true_w = np.random.normal(size=(num_features, num_classes)).astype('float32')

# Logit là điểm thô cho từng lớp; nhiễu nhỏ giúp bài toán bớt quá dễ.
# Phép nhân ma trận x @ true_w tạo điểm cho từng lớp của từng mẫu, rồi cộng nhiễu Gaussian nhỏ.
logits = x @ true_w + 0.25 * np.random.normal(size=(num_samples, num_classes))
# Chọn lớp có logit lớn nhất làm nhãn mục tiêu; int32 phù hợp với sparse_categorical_crossentropy.
y = np.argmax(logits, axis=1).astype('int32')

# Chia dữ liệu thành train và validation.
x_train, x_val = x[:1200], x[1200:]
y_train, y_val = y[:1200], y[1200:]

# Dữ liệu regression để thực hành custom metric RMSE.
x_reg = np.linspace(-3, 3, 1200).reshape(-1, 1).astype('float32')
y_reg = (2.5 * x_reg - 0.7 + 0.4 * np.sin(3 * x_reg)
         + 0.25 * np.random.normal(size=x_reg.shape)).astype('float32')
x_reg_train, x_reg_val = x_reg[:900], x_reg[900:]
y_reg_train, y_reg_val = y_reg[:900], y_reg[900:]

# Dữ liệu ticket giả lập: nhiều input, nhiều output.
ticket_samples = 1200

# title/body/tags mô phỏng ba nhóm đặc trưng khác nhau của ticket.
title_data = np.random.randint(0, 2, size=(ticket_samples, 100)).astype('float32')
body_data = np.random.randint(0, 2, size=(ticket_samples, 1000)).astype('float32')
tags_data = np.random.randint(0, 2, size=(ticket_samples, 12)).astype('float32')

# priority_targets là output hồi quy trong khoảng [0, 1].
priority_targets = (
    0.4 * title_data[:, :10].mean(axis=1)
    + 0.4 * body_data[:, :30].mean(axis=1)
    + 0.2 * tags_data[:, :4].mean(axis=1)
    + 0.05 * np.random.normal(size=(ticket_samples,))
).astype('float32')
priority_targets = np.clip(priority_targets, 0, 1).reshape(-1, 1)

# department_targets là output phân loại 4 lớp, sinh từ điểm của từng phòng ban.
department_scores = np.stack([
    title_data[:, :25].mean(axis=1) + tags_data[:, 0],
    body_data[:, :250].mean(axis=1) + tags_data[:, 1],
    body_data[:, 250:500].mean(axis=1) + tags_data[:, 2],
    title_data[:, 25:50].mean(axis=1) + tags_data[:, 3],
], axis=1)
department_targets = np.argmax(department_scores, axis=1).astype('int32')

# Tạo dictionary để truyền dữ liệu cho Functional API theo tên input/output.
ticket_split = 900
ticket_train_inputs = {
    'title': title_data[:ticket_split],
    'body': body_data[:ticket_split],
    'tags': tags_data[:ticket_split],
}
ticket_val_inputs = {
    'title': title_data[ticket_split:],
    'body': body_data[ticket_split:],
    'tags': tags_data[ticket_split:],
}
ticket_train_targets = {
    'priority': priority_targets[:ticket_split],
    'department': department_targets[:ticket_split],
}
ticket_val_targets = {
    'priority': priority_targets[ticket_split:],
    'department': department_targets[ticket_split:],
}

print('x_train:', x_train.shape, 'y_train:', y_train.shape)
print('x_val:', x_val.shape, 'y_val:', y_val.shape)
print('ticket title:', title_data.shape)
print('ticket priority:', priority_targets.shape)
print('ticket department:', department_targets.shape)

x_train: (1200, 20) y_train: (1200,)
x_val: (300, 20) y_val: (300,)
ticket title: (1200, 100)
ticket priority: (1200, 1)
ticket department: (1200,)


In [8]:
def plot_history(history, metric='accuracy'):
    """Vẽ metric train và validation từ object History của Keras.

    Tham số:
        history (keras.callbacks.History): object được trả về bởi model.fit().
        metric (str): tên metric cần vẽ, ví dụ 'accuracy' hoặc 'loss'.

    Kiểu trả về:
        None. Hàm hiển thị biểu đồ bằng matplotlib.
    """
    values = history.history

    # Nếu metric không tồn tại, in danh sách khóa để sinh viên tự kiểm tra tên đúng.
    if metric not in values:
        print(f'Không tìm thấy metric: {metric}')
        print('Các khóa hiện có:', sorted(values.keys()))
        return

    plt.figure(figsize=(6, 4))
    plt.plot(values[metric], label=metric)

    # Nếu có metric validation tương ứng, vẽ thêm để so sánh overfitting/underfitting.
    val_metric = 'val_' + metric
    if val_metric in values:
        plt.plot(values[val_metric], label=val_metric)

    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

# 7.1 A spectrum of workflows - Phổ các quy trình làm việc

Keras cho phép làm việc ở nhiều mức trừu tượng khác nhau. Bài tập đầu tiên yêu cầu sinh viên chọn mức workflow phù hợp với từng tình huống.

## Bài tập 1 - Chọn workflow phù hợp

Với mỗi tình huống, hãy chọn một trong các lựa chọn: `Sequential`, Functional API, model subclassing, custom training loop.

1. Cần tạo baseline nhanh cho bài toán phân loại vector một input, một output.
2. Cần xây mô hình xử lý ticket có ba input và hai output.
3. Cần kiến trúc có logic Python động trong forward pass.
4. Cần thuật toán huấn luyện đặc biệt, không thể mô tả bằng `fit()` thông thường.

### Câu trả lời của sinh viên

- **Tình huống 1:** 

    Bài toán đơn giản nhất có thể (1 input, 1 output, các lớp xếp chồng tuyến tính), dùng Sequential giúp viết code nhanh, gọn và cực kỳ tường minh.

- **Tình huống 2:** 

    Khi mô hình có nhiều input (multi-input) hoặc nhiều output (multi-output), Sequential không thể đáp ứng được. Functional API là lựa chọn hoàn hảo vì nó cho phép định nghĩa các đồ thị mạng (DAG) linh hoạt.

- **Tình huống 3:** 

    Khi cần các câu lệnh điều kiện mạng (if/else) hoặc vòng lặp (for) thay đổi động tùy theo dữ liệu đầu vào ngay trong quá trình lan truyền xuôi (call()), ta buộc phải subclass lại lớp keras.Model.

- **Tình huống 4:** 

    Khi các hàm loss, cơ chế cập nhật trọng số, hoặc các giải thuật (như GANs, Reinforcement Learning phức tạp) vượt quá khả năng cấu hình của hàm .fit(), ta phải tự viết vòng lặp huấn luyện bằng tf.GradientTape.

Giải thích ngắn: tại sao nên dùng mức trừu tượng cao nhất vẫn giải quyết được bài toán?

Việc sử dụng mức trừu tượng cao nhất (như Keras Model và fit) giúp chúng ta tập trung vào thiết kế kiến trúc và lựa chọn hàm mất mát, mà không phải lo lắng về chi tiết tính toán gradient và cập nhật trọng số.

# 7.2 Different ways to build Keras models - Các cách xây dựng mô hình Keras

## Bài tập 2 - Xây mô hình `Sequential`

Hoàn thiện hàm `make_sequential_classifier()` theo yêu cầu:

- Input shape là `(num_features,)`.
- Có ít nhất hai lớp `Dense` ẩn với activation `relu`.
- Có một lớp `Dropout` để giảm overfitting.
- Output có `num_classes` neuron và activation `softmax`.
- Compile bằng `Adam`, loss `sparse_categorical_crossentropy`, metric `accuracy`.
- Train 5 epoch và đánh giá trên tập validation.

In [10]:
def make_sequential_classifier():
    """Tạo mô hình phân loại bằng Sequential API.

    Tham số:
        Không có. Hàm dùng num_features và num_classes từ phần chuẩn bị dữ liệu.

    Kiểu trả về:
        keras.Sequential: model nhận tensor shape (batch_size, num_features)
        và trả về xác suất shape (batch_size, num_classes).
    """
    # TODO: thay phần raise bằng keras.Sequential([...], name='sequential_classifier')
    # Gợi ý layer: Input -> Dense(64, relu) -> Dropout(0.2)
    #              -> Dense(32, relu) -> Dense(num_classes, softmax)
    return keras.Sequential([
        layers.Input(shape=(num_features,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ])

    raise NotImplementedError('Hãy hoàn thiện make_sequential_classifier()')


seq_model = make_sequential_classifier()

# compile cấu hình optimizer, loss và metric cho training loop có sẵn của Keras.
seq_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

seq_model.summary()

# fit huấn luyện model và trả về History chứa log theo epoch.
history_seq = seq_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=32,
)

# evaluate đo loss và metric trên tập validation.
seq_eval = seq_model.evaluate(x_val, y_val, return_dict=True, verbose=0)
print(seq_eval)

# Các assert giúp kiểm tra nhanh lời giải của sinh viên.
assert isinstance(seq_model, keras.Model)
assert seq_model.output_shape[-1] == num_classes
assert 'accuracy' in history_seq.history
assert seq_eval['accuracy'] >= 0.55

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,523 (13.76 KB)

 Trainable params: 3,523 (13.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4817 - loss: 1.0333 - val_accuracy: 0.6633 - val_loss: 0.8397
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6442 - loss: 0.8099 - val_accuracy: 0.7800 - val_loss: 0.6503
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7783 - loss: 0.6096 - val_accuracy: 0.8367 - val_loss: 0.4978
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8208 - loss: 0.4824 - val_accuracy: 0.8800 - val_loss: 0.3873
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8633 - loss: 0.3816 - val_accuracy: 0.9000 - val_loss: 0.3219
{'accuracy': 0.8999999761581421, 'loss': 0.3218981921672821}


## Bài tập 3 - Functional API cho mô hình nhiều input, nhiều output

Hoàn thiện `build_ticket_model()` để xử lý bài toán ticket hỗ trợ khách hàng:

- Input `title`: vector 100 chiều.
- Input `body`: vector 1000 chiều.
- Input `tags`: vector 12 chiều.
- Dùng nhánh riêng cho `title` và `body`, sau đó nối với `tags`.
- Tạo layer trung gian tên `shared_features`.
- Output `priority`: 1 neuron, activation `sigmoid`.
- Output `department`: 4 neuron, activation `softmax`.

In [12]:
def build_ticket_model():
    """Xây mô hình ticket routing bằng Functional API.

    Tham số:
        Không có. Hàm dùng shape cố định của title, body và tags trong bài tập.

    Kiểu trả về:
        keras.Model: model có 3 input ('title', 'body', 'tags') và 2 output
        ('priority', 'department').
    """
    title_input = keras.Input(shape=(100,), name='title')
    body_input = keras.Input(shape=(1000,), name='body')
    tags_input = keras.Input(shape=(12,), name='tags')

    # TODO: xử lý title_input bằng Dense và đặt tên layer là 'title_features'
    title_features = layers.Dense(64, activation='relu', name='title_features')(title_input)
    # TODO: xử lý body_input bằng Dense và đặt tên layer là 'body_features'
    body_features = layers.Dense(64, activation='relu', name='body_features')(body_input)
    # TODO: nối các nhánh bằng layers.Concatenate(name='concatenate_features')
    concatenate_features = layers.Concatenate(name='concatenate_features')([title_features, body_features, tags_input])
    # TODO: tạo layer Dense trung gian tên 'shared_features'
    shared_features = layers.Dense(64, activation='relu', name='shared_features')(concatenate_features)
    # TODO: tạo hai output tên 'priority' và 'department'
    priority_output = layers.Dense(1, activation='linear', name='priority')(shared_features)
    department_output = layers.Dense(4, activation='softmax', name='department')(shared_features)

    return keras.Model(
        inputs=[title_input, body_input, tags_input],
        outputs=[priority_output, department_output],
        name='ticket_model',
    )

    raise NotImplementedError('Hãy hoàn thiện build_ticket_model()')

ticket_model = build_ticket_model()

# Mỗi output có loss và metric riêng vì priority là hồi quy còn department là phân loại.
ticket_model.compile(
    optimizer='rmsprop',
    loss={
        'priority': 'mse',
        'department': 'sparse_categorical_crossentropy',
    },
    metrics={
        'priority': ['mae'],
        'department': ['accuracy'],
    },
)

ticket_model.summary()

# Truyền dữ liệu bằng dictionary để Keras map đúng tên input/output.
history_ticket = ticket_model.fit(
    ticket_train_inputs,
    ticket_train_targets,
    validation_data=(ticket_val_inputs, ticket_val_targets),
    epochs=3,
    batch_size=32,
)

print('History keys:', sorted(history_ticket.history.keys()))
assert set(ticket_model.output_names) == {'priority', 'department'}
assert 'priority_mae' in history_ticket.history
assert 'department_accuracy' in history_ticket.history

Model: "ticket_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 100)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body (InputLayer)   │ (None, 1000)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_features      │ (None, 64)        │      6,464 │ title[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ body_features       │ (None, 64)        │     64,064 │ body[0][0]        │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tags (InputLayer)   │ (None, 12)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_featur… │ (None, 140)       │          0 │ title_features[0… │
│ (Concatenate)       │                   │            │ body_features[0]… │
│                     │                   │            │ tags[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_features     │ (None, 64)        │      9,024 │ concatenate_feat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ priority (Dense)    │ (None, 1)         │         65 │ shared_features[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department (Dense)  │ (None, 4)         │        260 │ shared_features[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 79,877 (312.02 KB)

 Trainable params: 79,877 (312.02 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - department_accuracy: 0.0000e+00 - department_loss: 1.4188 - loss: 1.6489 - priority_loss: 0.2240 - priority_mae: 1.3167 - val_department_accuracy: 0.0000e+00 - val_department_loss: 1.4127 - val_loss: 1.4647 - val_priority_loss: 0.0544 - val_priority_mae: 1.3651
Epoch 2/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - department_accuracy: 0.0000e+00 - department_loss: 1.3054 - loss: 1.3488 - priority_loss: 0.0342 - priority_mae: 1.2563 - val_department_accuracy: 0.0000e+00 - val_department_loss: 1.3604 - val_loss: 1.3921 - val_priority_loss: 0.0335 - val_priority_mae: 1.2751
Epoch 3/3
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - department_accuracy: 0.0000e+00 - department_loss: 1.1958 - loss: 1.2404 - priority_loss: 0.0294 - priority_mae: 1.2473 - val_department_accuracy: 0.0000e+00 - val_department_loss: 1.3286 - val_loss: 1.3516 - val_priority_loss: 0.0264 - val_priority_mae: 1.3093
History keys: ['department_accuracy', 'department_loss', 'los

## Bài tập 4 - Inspect model và lấy đặc trưng trung gian

Functional API lưu được đồ thị kết nối của mô hình. Vì vậy ta có thể inspect layer và tạo model mới lấy output trung gian.

Yêu cầu:

- In tên và output shape của từng layer trong `ticket_model`.
- Tạo `feature_extractor` lấy output của layer `shared_features`.
- Chạy thử trên 3 mẫu validation và kiểm tra shape.

In [14]:
for layer in ticket_model.layers:
    # output_shape giúp kiểm tra kích thước tensor đi qua từng layer.
    output_shape = getattr(layer, 'output_shape', 'dynamic')
    print(f'{layer.name:25s} | {output_shape}')

# TODO: tạo model lấy output của layer 'shared_features'
# Gợi ý: dùng keras.Model(inputs=ticket_model.inputs, outputs=...)
feature_extractor = keras.Model(
    inputs=ticket_model.inputs, 
    outputs=ticket_model.get_layer('shared_features').output, 
    name='feature_extractor'
)

# Lấy đặc trưng trung gian cho 3 mẫu validation đầu tiên.
sample_features = feature_extractor.predict(
    {k: v[:3] for k, v in ticket_val_inputs.items()},
    verbose=0,
)

print('Shape đặc trưng trung gian:', sample_features.shape)
assert sample_features.shape[0] == 3

title                     | dynamic
body                      | dynamic
title_features            | dynamic
body_features             | dynamic
tags                      | dynamic
concatenate_features      | dynamic
shared_features           | dynamic
priority                  | dynamic
department                | dynamic
Shape đặc trưng trung gian: (3, 64)


### Câu trả lời của sinh viên

1. Vì sao Functional API thuận tiện hơn `Sequential` khi mô hình có nhiều input/output?

    Functional API cho phép định nghĩa rõ ràng nhiều input và output bằng cách đặt tên, trong khi Sequential chỉ hỗ trợ một input và một output duy nhất. Với Functional API, ta có thể dễ dàng xây dựng các mô hình phức tạp như multi-branch hoặc multi-task, còn Sequential chỉ phù hợp với mô hình dạng chuỗi tuyến tính.

2. Vì sao Functional API dễ inspect hơn model subclassing?

    Functional API tạo ra một graph tĩnh của mô hình, cho phép truy cập dễ dàng đến các layer và tensor thông qua tên. Trong khi đó, model subclassing định nghĩa forward pass trong phương thức call(), làm cho cấu trúc mô hình trở nên động và khó truy cập trực tiếp đến các layer hoặc tensor trung gian mà không chạy qua toàn bộ forward pass.

3. Khi nào việc lấy đặc trưng trung gian có ích trong thực tế?

    Trong thực tế việc lấy đặc trưng trung gian rất hữu ích khi ta muốn sử dụng lại phần đã học của một mô hình cho các tác vụ khác, như fine-tuning, transfer learning, hoặc khi muốn phân tích và trực quan hóa cách mô hình xử lý dữ liệu. Ví dụ, trong bài toán ticket routing, ta có thể muốn sử dụng đặc trưng từ layer 'shared_features' để huấn luyện một mô hình khác chuyên sâu hơn cho một tác vụ cụ thể như dự đoán độ ưu tiên mà không cần phải huấn luyện lại toàn bộ mô hình.

## Bài tập 5 - Layer sharing

Trong Functional API, nếu cùng một layer object được gọi nhiều lần, các lần gọi đó dùng chung trọng số. Hoàn thiện mô hình đơn giản bên dưới để tính độ tương tự giữa hai vector.

In [15]:
def build_siamese_model():
    """Xây mô hình siamese nhỏ để minh họa layer sharing.

    Tham số:
        Không có. Hai input đều có shape (10,).

    Kiểu trả về:
        tuple[keras.Model, keras.layers.Layer]: gồm model tính cosine similarity
        và layer shared_encoder để kiểm tra weight được chia sẻ.
    """
    input_a = keras.Input(shape=(10,), name='item_a')
    input_b = keras.Input(shape=(10,), name='item_b')

    shared_encoder = layers.Dense(16, activation='relu', name='shared_encoder')

    # TODO: gọi cùng shared_encoder trên input_a và input_b
    encoded_a = shared_encoder(input_a)
    encoded_b = shared_encoder(input_b)
    # TODO: dùng layers.Dot(axes=1, normalize=True, name='cosine_similarity')
    cosine_similarity = layers.Dot(axes=1, normalize=True, name='cosine_similarity')([encoded_a, encoded_b])
    # TODO: trả về model và shared_encoder
    return keras.Model(
        inputs=[input_a, input_b],
        outputs=cosine_similarity,
        name='siamese_model'), shared_encoder
    
    raise NotImplementedError('Hãy hoàn thiện build_siamese_model()')

siamese_model, shared_encoder = build_siamese_model()
siamese_model.summary()

# Tạo 5 cặp vector để kiểm tra output của mô hình siamese.
pairs_a = np.random.normal(size=(5, 10)).astype('float32')
pairs_b = np.random.normal(size=(5, 10)).astype('float32')
scores = siamese_model.predict([pairs_a, pairs_b], verbose=0)
print(scores)

assert len(shared_encoder.weights) == 2
assert scores.shape == (5, 1)

Model: "siamese_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ item_a (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_b (InputLayer) │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_encoder      │ (None, 16)        │        176 │ item_a[0][0],     │
│ (Dense)             │                   │            │ item_b[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)         │          0 │ shared_encoder[0… │
│ (Dot)               │                   │            │ shared_encoder[1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 176 (704.00 B)

 Trainable params: 176 (704.00 B)

 Non-trainable params: 0 (0.00 B)

[[0.5031366 ]
 [0.20356975]
 [0.5322474 ]
 [0.1653554 ]
 [0.8310343 ]]


## Bài tập 6 - Model subclassing

Hoàn thiện lớp `MLPClassifier`. Chú ý truyền tham số `training` cho `Dropout` trong `call()` để model hoạt động đúng giữa training và inference.

In [18]:
class MLPClassifier(keras.Model):
    """Mô hình MLP phân loại nhiều lớp viết bằng subclassing.

    Tham số:
        num_classes (int): số lớp cần dự đoán.

    Kiểu trả về khi gọi model:
        tf.Tensor: xác suất dự đoán shape (batch_size, num_classes).
    """

    def __init__(self, num_classes):
        """Khai báo layer dùng trong forward pass."""
        super().__init__(name='subclassed_mlp')
        # TODO: khai báo các layer cần dùng trong forward pass
        # Gợi ý: Dense(64, relu), Dropout(0.3), Dense(32, relu), Dense(num_classes, softmax)
        self.dense1 = layers.Dense(64, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.dense2 = layers.Dense(32, activation='relu')
        self.classifier = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        """Chạy forward pass của model.

        Tham số:
            inputs (tf.Tensor): batch dữ liệu đầu vào.
            training (bool): True khi train, False khi inference.

        Kiểu trả về:
            tf.Tensor: xác suất dự đoán của từng lớp.
        """
        # TODO: viết forward pass và truyền training cho Dropout
        x = self.dense1(inputs)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        return self.classifier(x)

sub_model = MLPClassifier(num_classes=num_classes)

# Gọi model một lần để tạo weights trước khi summary.
_ = sub_model(tf.zeros((1, num_features)))
sub_model.summary()

sub_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_sub = sub_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=32,
)

assert isinstance(sub_model, keras.Model)

Model: "subclassed_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (1, 64)                │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (1, 32)                │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (1, 3)                 │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,523 (13.76 KB)

 Trainable params: 3,523 (13.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4183 - loss: 1.1047 - val_accuracy: 0.5767 - val_loss: 0.9456
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6075 - loss: 0.8759 - val_accuracy: 0.7067 - val_loss: 0.7694
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.7298 - val_accuracy: 0.7900 - val_loss: 0.6156
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7758 - loss: 0.5897 - val_accuracy: 0.8500 - val_loss: 0.4862
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8142 - loss: 0.4909 - val_accuracy: 0.8667 - val_loss: 0.4021


### Câu trả lời của sinh viên

1. Ưu điểm lớn nhất của model subclassing là gì?
2. Hạn chế của subclassing so với Functional API là gì?
3. Trong `call()`, vì sao nên có tham số `training=False`?

# 7.3 Using built-in training and evaluation loops - Dùng vòng lặp huấn luyện và đánh giá có sẵn

## Bài tập 7 - `compile()`, `fit()`, `evaluate()`, `predict()` và `History`

Dùng `seq_model` đã huấn luyện ở Bài tập 2 để:

- Đánh giá lại trên tập validation.
- Dự đoán nhãn cho 8 mẫu đầu tiên.
- Vẽ biểu đồ accuracy và loss theo epoch.
- Nhận xét dấu hiệu overfitting hoặc underfitting nếu có.

In [ ]:
# evaluate trả về loss và metric trên validation set.
eval_result = seq_model.evaluate(x_val, y_val, return_dict=True, verbose=0)
print('Validation:', eval_result)

# predict trả về xác suất cho từng lớp; argmax chuyển xác suất thành nhãn dự đoán.
pred_probs = seq_model.predict(x_val[:8], verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)

print('Pred labels:', pred_labels)
print('True labels:', y_val[:8])

# Vẽ metric train/validation để quan sát quá trình học.
plot_history(history_seq, 'accuracy')
plot_history(history_seq, 'loss')

### Câu trả lời của sinh viên

1. `fit()` trả về object gì và object đó dùng để làm gì?

    fit() trả về một object thuộc lớp keras.callbacks.History nhằm lưu trữ giá trị của các metric và loss theo từng epoch trong quá trình huấn luyện.

2. `evaluate()` khác `predict()` như thế nào?

    evaluate() trả về loss và metric trên tập dữ liệu, thường dùng để đánh giá hiệu suất của model. predict() trả về dự đoán của model (thường là xác suất hoặc nhãn) cho từng mẫu đầu vào, dùng để kiểm tra kết quả dự đoán cụ thể.

3. Nếu training accuracy tăng nhưng validation accuracy giảm, bạn sẽ nghĩ đến vấn đề gì?

    Nếu training accuracy tăng nhưng validation accuracy giảm, đó có thể là dấu hiệu của overfitting, tức là model học quá kỹ vào dữ liệu huấn luyện và không tổng quát tốt trên dữ liệu mới (validation).

## Bài tập 8 - Tự viết custom metric

Hoàn thiện metric `RootMeanSquaredError` theo cấu trúc của `keras.metrics.Metric`:

- `update_state()`: cập nhật tổng bình phương sai số và số lượng phần tử.
- `result()`: trả về căn bậc hai của MSE.
- `reset_state()`: đưa metric về trạng thái ban đầu.

In [19]:
class RootMeanSquaredError(keras.metrics.Metric):
    """Metric RMSE tự viết theo chuẩn Keras Metric.

    Tham số:
        name (str): tên metric hiển thị trong log.
        **kwargs: tham số bổ sung do Keras truyền vào.

    Kiểu trả về của result:
        tf.Tensor: giá trị RMSE hiện tại.
    """

    def __init__(self, name='rmse', **kwargs):
        """Khởi tạo các biến trạng thái dùng để tích lũy sai số."""
        super().__init__(name=name, **kwargs)
        self.squared_sum = self.add_weight(name='squared_sum', initializer='zeros')
        self.total = self.add_weight(name='total', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        """Cập nhật metric bằng một batch dự đoán.

        Tham số:
            y_true (tf.Tensor): giá trị thật.
            y_pred (tf.Tensor): giá trị model dự đoán.
            sample_weight (tf.Tensor | None): trọng số mẫu, không bắt buộc trong bài này.

        Kiểu trả về:
            None. Hàm cập nhật trực tiếp biến trạng thái của metric.
        """
        y_true = tf.cast(y_true, y_pred.dtype)
        # TODO: tính squared_errors = (y_true - y_pred)^2
        squared_errors = tf.reduce_sum(tf.square(y_true - y_pred))
        batch_size = tf.cast(tf.shape(y_true)[0], self.total.dtype)
        # TODO: cộng tổng squared_errors vào self.squared_sum
        self.squared_sum.assign_add(squared_errors)
        # TODO: cộng số phần tử vào self.total
        self.total.assign_add(batch_size)

    def result(self):
        """Trả về RMSE từ trạng thái đã tích lũy."""
        # TODO: trả về sqrt(squared_sum / total)
        return tf.sqrt(self.squared_sum / self.total)
        raise NotImplementedError('Hãy hoàn thiện result()')

    def reset_state(self):
        """Reset metric sau mỗi epoch hoặc trước một lần đánh giá mới."""
        self.squared_sum.assign(0.0)
        self.total.assign(0.0)


metric_test = RootMeanSquaredError()
metric_test.update_state(tf.constant([[1.0], [2.0]]), tf.constant([[1.0], [4.0]]))
print('RMSE test:', float(metric_test.result()))
assert np.isclose(float(metric_test.result()), np.sqrt(2.0), atol=1e-5)

# Model regression nhỏ để kiểm tra custom metric khi dùng trong compile/fit.
reg_model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])

reg_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=[RootMeanSquaredError()],
)

history_reg = reg_model.fit(
    x_reg_train,
    y_reg_train,
    validation_data=(x_reg_val, y_reg_val),
    epochs=5,
    batch_size=32,
)

RMSE test: 1.4142135381698608
Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 16.3398 - rmse: 4.0423 - val_loss: 16.3579 - val_rmse: 4.0445
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13.6079 - rmse: 3.6889 - val_loss: 8.0549 - val_rmse: 2.8381
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 9.2079 - rmse: 3.0345 - val_loss: 3.2186 - val_rmse: 1.7941
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.8634 - rmse: 1.9655 - val_loss: 2.0141 - val_rmse: 1.4192
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5986 - rmse: 0.7737 - val_loss: 1.8206 - val_rmse: 1.3493


## Bài tập 9 - Callbacks

Sử dụng callback để điều khiển quá trình huấn luyện:

- `ModelCheckpoint`: lưu model tốt nhất theo `val_loss`.
- `EarlyStopping`: dừng sớm nếu `val_loss` không cải thiện.
- Callback tự viết: in tóm tắt mỗi epoch và tự dừng nếu `val_accuracy` vượt ngưỡng do bạn chọn.

In [28]:
class EpochSummaryCallback(keras.callbacks.Callback):
    """Callback tự viết để in log và có thể dừng sớm theo val_accuracy.

    Tham số:
        target_val_accuracy (float): ngưỡng validation accuracy mong muốn.

    Kiểu trả về:
        None. Callback tác động lên quá trình train thông qua self.model.
    """

    def __init__(self, target_val_accuracy=0.92):
        super().__init__()
        self.target_val_accuracy = target_val_accuracy

    def on_epoch_end(self, epoch, logs=None):
        """Được Keras gọi sau mỗi epoch.

        Tham số:
            epoch (int): chỉ số epoch bắt đầu từ 0.
            logs (dict | None): dictionary chứa loss và metric của epoch.

        Kiểu trả về:
            None.
        """
        logs = logs or {}
        # TODO: in epoch, loss, accuracy, val_loss, val_accuracy
        epoch_num = epoch + 1
        loss = logs.get('loss', 'N/A')
        accuracy = logs.get('accuracy', 'N/A')
        val_loss = logs.get('val_loss', 'N/A')
        val_accuracy = logs.get('val_accuracy', 'N/A')
        # TODO: nếu val_accuracy >= target_val_accuracy thì đặt self.model.stop_training = True
        if val_accuracy >= self.target_val_accuracy:
            self.model.stop_training = True


checkpoint_dir = 'tmp_c7_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'best_seq_model.keras')

callback_model = make_sequential_classifier()
callback_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    # Lưu model tốt nhất theo validation loss.
    keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
    ),
    # Dừng sớm nếu validation loss không cải thiện sau 3 epoch.
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    EpochSummaryCallback(target_val_accuracy=0.92),
]

history_callback = callback_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
)

assert os.path.exists(checkpoint_path)

Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3767 - loss: 1.1445 - val_accuracy: 0.6333 - val_loss: 0.9296
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6442 - loss: 0.8484 - val_accuracy: 0.7533 - val_loss: 0.7154
Epoch 3/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7508 - loss: 0.6551 - val_accuracy: 0.8033 - val_loss: 0.5468
Epoch 4/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8208 - loss: 0.5126 - val_accuracy: 0.8533 - val_loss: 0.4263
Epoch 5/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8608 - loss: 0.4042 - val_accuracy: 0.8800 - val_loss: 0.3399
Epoch 6/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8800 - loss: 0.3364 - val_accuracy: 0.8867 - val_loss: 0.2914
Epoch 7/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8917 - loss: 0.2897 - val_accuracy: 0.9000 - val_loss: 0.2612
Epoch 8/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9033 - loss: 0.2586 - val_accuracy: 0.9067 - val_loss:

### Câu trả lời của sinh viên

1. Callback khác gì với code viết sau khi `fit()` kết thúc?

    Callback là một cách để can thiệp vào quá trình huấn luyện ngay trong khi nó đang diễn ra, cho phép bạn thực hiện các hành động như lưu model, điều chỉnh learning rate, hoặc dừng sớm dựa trên các điều kiện nhất định. Trong khi đó, code viết sau `fit()` chỉ được thực thi sau khi toàn bộ quá trình huấn luyện đã kết thúc, và không thể ảnh hưởng đến quá trình đó.

2. Khi nào nên dùng `EarlyStopping`?

    EarlyStopping hữu ích khi tránh overfitting bằng cách dừng huấn luyện ngay khi hiệu suất trên validation set không còn cải thiện. Điều này giúp tiết kiệm thời gian và tài nguyên, đồng thời thường dẫn đến mô hình có khả năng tổng quát hóa tốt hơn trên dữ liệu mới.

3. Vì sao `restore_best_weights=True` thường hữu ích?

    'restore_best_weights=True' giúp đảm bảo rằng sau khi dừng sớm, mô hình sẽ được khôi phục về trạng thái tốt nhất đã từng đạt được trên validation set, thay vì giữ lại weights của epoch cuối cùng có thể đã bị overfitting. Điều này thường cải thiện hiệu suất của mô hình trên dữ liệu unseen.

# 7.4 Writing your own training and evaluation loops - Tự viết vòng lặp huấn luyện và đánh giá

## Bài tập 10 - Custom training loop với `GradientTape`

Hoàn thiện vòng lặp huấn luyện thủ công. Chú ý:

- Gọi model với `training=True` khi train.
- Dùng `tape.gradient()` để tính gradient.
- Dùng optimizer để cập nhật `trainable_weights`.
- Cập nhật metric sau mỗi batch và reset metric sau mỗi epoch.

In [21]:
loop_model = make_sequential_classifier()
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy()
train_acc_metric = keras.metrics.SparseCategoricalAccuracy()

# Dataset được shuffle và chia batch để dùng trong custom training loop.
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=len(x_train))
    .batch(32)
)

for epoch in range(3):
    # Reset metric để mỗi epoch có accuracy riêng.
    train_acc_metric.reset_state()

    for x_batch, y_batch in train_dataset:
        # GradientTape ghi lại phép tính forward để lấy đạo hàm của loss.
        with tf.GradientTape() as tape:
            y_pred = loop_model(x_batch, training=True)
            loss_value = loss_fn(y_batch, y_pred)

        # TODO: tính gradients bằng tape.gradient
        gradients = tape.gradient(loss_value, loop_model.trainable_weights)
        # TODO: cập nhật weights bằng optimizer.apply_gradients
        optimizer.apply_gradients(zip(gradients, loop_model.trainable_weights))
        # TODO: cập nhật train_acc_metric
        train_acc_metric.update_state(y_batch, y_pred)

    print(f'Epoch {epoch + 1}: accuracy = {train_acc_metric.result().numpy():.4f}')

Epoch 1: accuracy = 0.4392
Epoch 2: accuracy = 0.6667
Epoch 3: accuracy = 0.7667


## Bài tập 11 - Evaluation loop và `tf.function`

Hoàn thiện `train_step()` được biên dịch bằng `tf.function`, sau đó viết evaluation loop dùng `training=False`.

In [22]:
fast_model = make_sequential_classifier()
fast_optimizer = keras.optimizers.Adam(learning_rate=1e-3)
fast_loss_fn = keras.losses.SparseCategoricalCrossentropy()
fast_train_acc = keras.metrics.SparseCategoricalAccuracy()
fast_val_acc = keras.metrics.SparseCategoricalAccuracy()

# Tạo dataset cho train và validation.
fast_train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=len(x_train))
    .batch(32)
)
fast_val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(32)


@tf.function
def train_step(x_batch, y_batch):
    """Thực hiện một bước train đã được TensorFlow biên dịch.

    Tham số:
        x_batch (tf.Tensor): batch đặc trưng train.
        y_batch (tf.Tensor): batch nhãn train.

    Kiểu trả về:
        tf.Tensor: loss của batch hiện tại.
    """
    with tf.GradientTape() as tape:
        y_pred = fast_model(x_batch, training=True)
        loss_value = fast_loss_fn(y_batch, y_pred)

    # TODO: tính gradient, cập nhật weight, cập nhật fast_train_acc
    gradients = tape.gradient(loss_value, fast_model.trainable_weights)
    fast_optimizer.apply_gradients(zip(gradients, fast_model.trainable_weights))
    fast_train_acc.update_state(y_batch, y_pred)
    return loss_value


for epoch in range(3):
    fast_train_acc.reset_state()
    fast_val_acc.reset_state()

    for x_batch, y_batch in fast_train_dataset:
        train_step(x_batch, y_batch)

    for x_batch, y_batch in fast_val_dataset:
        # training=False để tắt hành vi dành riêng cho training như Dropout.
        val_pred = fast_model(x_batch, training=False)
        # TODO: cập nhật fast_val_acc

    print(
        f'Epoch {epoch + 1}: '
        f'train_acc={fast_train_acc.result().numpy():.4f}, '
        f'val_acc={fast_val_acc.result().numpy():.4f}'
    )

Epoch 1: train_acc=0.3825, val_acc=0.0000
Epoch 2: train_acc=0.6192, val_acc=0.0000
Epoch 3: train_acc=0.7308, val_acc=0.0000


## Bài tập 12 - Tận dụng `fit()` với `train_step()` tùy biến

Thay vì tự viết toàn bộ vòng lặp, ta có thể tùy biến logic của một bước train bằng cách override `train_step()` nhưng vẫn dùng được `fit()`, callbacks, metrics và validation.

In [27]:
class CustomTrainStepModel(keras.Model):
    """Model override train_step nhưng vẫn dùng được fit của Keras.

    Tham số:
        *args, **kwargs: giống keras.Model, thường gồm inputs, outputs và name.

    Kiểu trả về của train_step:
        dict: tên metric ánh xạ tới giá trị hiện tại để Keras ghi log.
    """

    def train_step(self, data):
        """Định nghĩa logic huấn luyện cho một batch.

        Tham số:
            data (tuple): gồm x_batch và y_batch do fit truyền vào.

        Kiểu trả về:
            dict: kết quả metric hiện tại.
        """
        x_batch, y_batch = data

        with tf.GradientTape() as tape:
            y_pred = self(x_batch, training=True)
            # TODO: tính loss bằng self.compute_loss(...)
            loss_value = self.compute_loss(x_batch, y_batch, y_pred)

        # TODO: tính gradients và cập nhật optimizer
        gradients = tape.gradient(loss_value, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        # TODO: cập nhật metrics trong self.metrics
        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(loss_value)
            else:
                metric.update_state(y_batch, y_pred)
        # TODO: return dict {metric_name: metric_result}
        return {metric.name: metric.result() for metric in self.metrics}


inputs = keras.Input(shape=(num_features,), name='features')
x_custom = layers.Dense(64, activation='relu')(inputs)
x_custom = layers.Dense(32, activation='relu')(x_custom)
outputs = layers.Dense(num_classes, activation='softmax')(x_custom)

custom_fit_model = CustomTrainStepModel(inputs, outputs, name='custom_train_step_model')
custom_fit_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# fit vẫn xử lý epoch, batch, validation và callbacks; chỉ logic train_step được tùy biến.
history_custom_fit = custom_fit_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=3,
    batch_size=32,
)

Epoch 1/3
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4392 - loss: 1.0482 - val_accuracy: 0.6367 - val_loss: 0.9017
Epoch 2/3
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7350 - loss: 0.7616 - val_accuracy: 0.7733 - val_loss: 0.6627
Epoch 3/3
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8533 - loss: 0.5195 - val_accuracy: 0.8467 - val_loss: 0.4635


### Câu trả lời của sinh viên

1. `GradientTape` dùng để làm gì?

    GradientTape là công cụ của TensorFlow để ghi lại các phép tính trong forward pass, cho phép tính toán đạo hàm (gradient) của loss đối với các biến trainable. Để cập nhật weights trong quá trình huấn luyện bằng cách sử dụng optimizer.

2. `tf.function` giúp gì cho performance và gây khó khăn gì khi debug?

    tf.function là một decorator của TensorFlow để biên dịch một hàm Python thành một biểu đồ tính toán (computation graph) hiệu quả. Điều này giúp tăng tốc độ thực thi, đặc biệt trên GPU/TPU. Tuy nhiên, khi debug, tf.function có thể làm mất đi thông tin lỗi chi tiết và stack trace, khiến việc tìm lỗi trở nên khó khăn hơn.

3. So sánh custom training loop hoàn toàn với việc override `train_step()` rồi dùng `fit()`.

    Custom training loop hoàn toàn (sử dụng GradientTape và vòng lặp thủ công) cho phép kiểm soát tối đa quá trình huấn luyện, nhưng đòi hỏi viết nhiều code hơn và tự xử lý logic như epoch, batch, validation, và metric. Override train_step() vẫn cho phép tùy biến logic huấn luyện nhưng tận dụng được các tiện ích của fit() như quản lý epoch, batch, validation, và callbacks, giúp code ngắn gọn hơn.

4. Khi inference, vì sao nên gọi model với `training=False`?

    Khi inference, gọi model với training=False giúp tắt các hành vi đặc biệt dành cho training như Dropout và BatchNormalization. Điều này đảm bảo rằng mô hình hoạt động ổn định và cho kết quả dự đoán nhất quán, thay vì có sự ngẫu nhiên hoặc biến động do các

# Tổng kết và yêu cầu nộp bài

Trước khi nộp, hãy kiểm tra:

- Đã hoàn thiện tất cả phần `TODO`.
- Các model chính train được và có output trong notebook.
- Các câu hỏi lý thuyết đã được trả lời bằng tiếng Việt.
- Có ít nhất một biểu đồ history.
- Có phần so sánh ngắn giữa `Sequential`, Functional API, subclassing và custom loop.

Câu hỏi mở rộng: Trong một dự án deep learning thực tế, bạn sẽ bắt đầu bằng workflow nào trước? Khi nào bạn mới chuyển sang mức tùy biến thấp hơn?